# Careerjet job scraping

Try the Careerjet API first, then fall back to Selenium scraping.

## 1. Imports

In [30]:
import httpx
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import sqlite3, time

In [31]:
url = "https://www.careerjet.it/offerte-lavoro"
params = {"s": "python", "l": "Italia"}
headers = {"User-Agent": "Mozilla/5.0"}

## 3. Fallback: scrape with Selenium

In [32]:
# Env settings needed on unix system with wayland issues

import os
os.environ['GDK_BACKEND']='x11'
os.environ['DISABLE_WAYLAND']='1'
os.environ['GTK_USE_PORTAL']='0'
os.environ['GSETTINGS_BACKEND']='memory'

In [33]:
options = webdriver.ChromeOptions()
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
wait = WebDriverWait(driver, 10)
jobs = []
for page in range(1, 16):
    url = f"https://www.careerjet.it/offerte-lavoro?s=python&l=Italia&p={page}"
    driver.get(url)
    time.sleep(3)

    job_cards = wait.until(EC.presence_of_all_elements_located((
        By.CSS_SELECTOR, "article, .job, .job-listing, [data-jobid], .listing"
    )))

    for card in job_cards:
        try:
            title_el = card.find_element(By.CSS_SELECTOR, "h2 a, h3 a, .title a, .job-title a, a")
            title = title_el.text.strip()
            link = title_el.get_attribute("href")
        except Exception:
            title, link = "", ""
        try:
            company = card.find_element(By.CSS_SELECTOR, ".company, .recruiter, [itemprop='hiringOrganization']").text.strip()
        except Exception:
            company = ""
        try:
            location = card.find_element(By.CSS_SELECTOR, ".location, [itemprop='jobLocation'], .place, .city").text.strip()
        except Exception:
            location = ""
        try:
            salary = card.find_element(By.CSS_SELECTOR, ".salary, .pay, [itemprop='baseSalary'], .wage, .compensation").text.strip()
        except Exception:
            salary = ""
        try:
            description = card.find_element(By.CSS_SELECTOR, ".description, .job-description, .summary, .snippet, p, [itemprop='description']").text.strip()
        except Exception:
            description = ""
        jobs.append({
            "title": title,
            "company": company,
            "location": location,
            "salary": salary,
            "description": description,
            "link": link,
        })

if not jobs:
    print("No job postings found across 15 pages")
    df = pd.DataFrame(columns=["title", "company", "location", "salary_min", "salary_max", "description", "link"])
else:
    df = pd.DataFrame(jobs)
    print(f"Found {len(df)} job postings across 15 pages")

    import re

    def parse_salary(s):
        if not s:
            return None
        s_lower = s.lower()
        # detect period: monthly vs yearly
        monthly_keywords = ["mese", "month", "al mese", "/mese", "mensile", "per mese", "mensili"]
        yearly_keywords = ["anno", "year", "annuo", "per anno", "annually", "/anno", "annuale"]
        is_monthly = any(k in s_lower for k in monthly_keywords)
        is_yearly = any(k in s_lower for k in yearly_keywords)
        # normalize: treat k/K as *1000, remove currency symbols and dots (thousands sep)
        t = s_lower.replace("€", "").replace("$", "").replace("usd", "").replace("eur", "")
        t = re.sub(r"[.,]", "", t)  # remove dots/commas used as thousand separators
        t = re.sub(r"(\d+)\s*k\\b", lambda m: str(int(m.group(1)) * 1000), t)
        nums = [int(x) for x in re.findall(r"\d+", t)]
        if not nums:
            return None
        if len(nums) == 1:
            val = nums[0]
        else:
            val = (min(nums) + max(nums)) / 2  # use midpoint of range
        # convert monthly to yearly
        if is_monthly and not is_yearly:
            val = val * 12
        return val

    salary_parsed = df["salary"].apply(parse_salary)
    df["salary_min"] = salary_parsed
    df["salary_max"] = salary_parsed
    df = df.drop(columns=["salary"])

    print(df.head())



Found 300 job postings across 15 pages
                                            title          company  \
0                     Python Full Stack Developer  Energent S.p.A.   
1                     Python Full Stack Developer  Energent S.p.A.   
2                  PYTHON DEVELOPER - FULL REMOTE       LOG.IT srl   
3  Senior Full Stack Developer (Python / Angular)       Dedalo SRL   
4                        Python Software Engineer        DID s.r.l   

                       location      description  \
0                          Roma  Energent S.p.A.   
1                          Roma  Energent S.p.A.   
2                        Milano       LOG.IT srl   
3             Carinaro, Caserta       Dedalo SRL   
4  Provincia di Monza e Brianza        DID s.r.l   

                                                link  salary_min  salary_max  
0  https://www.careerjet.it/jobad/itcf7686a7f5f83...     33000.0     33000.0  
1  https://www.careerjet.it/jobad/it30788eb56266d...     27500.0     

In [34]:
# Save to SQLite with upsert (insert new, update existing by link)
conn = sqlite3.connect("job_postings.db")
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS careerjet (
        title TEXT,
        company TEXT,
        location TEXT,
        salary_min INTEGER,
        salary_max INTEGER,
        description TEXT,
        link TEXT UNIQUE
    )
""")

rows = [
    (
        r.get("title", ""),
        r.get("company", ""),
        r.get("location", ""),
        r.get("salary_min"),
        r.get("salary_max"),
        r.get("description", ""),
        r.get("link", ""),
    )
    for _, r in df.iterrows()
]

cur.executemany("""
    INSERT INTO careerjet (title, company, location, salary_min, salary_max, description, link)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    ON CONFLICT(link) DO UPDATE SET
        title = excluded.title,
        company = excluded.company,
        location = excluded.location,
        salary_min = excluded.salary_min,
        salary_max = excluded.salary_max,
        description = excluded.description
""", rows)

conn.commit()
conn.close()
print(f"Upserted {len(rows)} rows into job_postings.db table 'careerjet'")


Upserted 300 rows into job_postings.db table 'careerjet'
